# Typhoon OCR — Election Vote Extraction Pipeline v3

| Layer | หน้าที่ |
|-------|--------|
| **1 — Setup** | Install dependencies, mount Drive, API keys |
| **2 — OCR** | Typhoon API → บันทึก `.text` files |
| **3 — Reasoning** | DeepSeek V3.2 อ่าน text → `Id, Vote` CSV |
| **4 — Submission** | Merge CSV กับ `submission_template.csv` |

---

## Layer 1 — Environment Setup


In [1]:
# ── 1.1 Install Dependencies ──────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

!pip install -q requests pandas tqdm "openai>=1.0.0" --upgrade
print("✅ Dependencies ready")

✅ Dependencies ready


In [2]:
# ── 1.2 Mount Google Drive & Config ───────────────────────────────────
from google.colab import drive, userdata
import os

drive.mount('/content/drive')

ROOT = "/content/drive/MyDrive/Typhoon_OCR_Data"

# API Keys
TYPHOON_API_KEY = None
DEEPSEEK_API_KEY = None

try:
    TYPHOON_API_KEY = userdata.get('TYPHOON_API_KEY')
    if TYPHOON_API_KEY:
        print("✅ TYPHOON_API_KEY loaded")
    else:
        print("❌ TYPHOON_API_KEY not found — กรุณาตั้งค่าใน Colab Secrets")
except Exception as e:
    print(f"❌ Typhoon key error: {e}")

try:
    DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')
    if DEEPSEEK_API_KEY:
        print("✅ DEEPSEEK_API_KEY loaded")
    else:
        print("❌ DEEPSEEK_API_KEY not found — กรุณาตั้งค่าใน Colab Secrets")
except Exception as e:
    print(f"❌ DeepSeek key error: {e}")

# Verify structure
for d in ['images', 'sample_labels']:
    path = os.path.join(ROOT, d)
    print(f"  {'[FOUND]' if os.path.isdir(path) else '[MISSING]'} {d}/")

template_path = os.path.join(ROOT, 'submission_template.csv')
print(f"  {'[FOUND]' if os.path.isfile(template_path) else '[MISSING]'} submission_template.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ TYPHOON_API_KEY loaded
✅ DEEPSEEK_API_KEY loaded
  [FOUND] images/
  [FOUND] sample_labels/
  [FOUND] submission_template.csv


In [3]:
# ── 1.3 Directory Setup ───────────────────────────────────────────────
DIRS = {
    'images':  os.path.join(ROOT, 'images'),
    'textlog': os.path.join(ROOT, 'Textlog'),
    'result':  os.path.join(ROOT, 'Result'),
}

for name, path in DIRS.items():
    os.makedirs(path, exist_ok=True)

os.makedirs(os.path.join(DIRS['textlog'], 'constituency'), exist_ok=True)
os.makedirs(os.path.join(DIRS['textlog'], 'party_list'), exist_ok=True)

print("✅ Directories ready")
print(f"   Textlog → {DIRS['textlog']}")
print(f"   Result  → {DIRS['result']}")

✅ Directories ready
   Textlog → /content/drive/MyDrive/Typhoon_OCR_Data/Textlog
   Result  → /content/drive/MyDrive/Typhoon_OCR_Data/Result


In [4]:
# ── 1.4 Import Libraries ─────────────────────────────────────────────
import re
import glob
import json
import time
import requests
import pandas as pd
from collections import defaultdict
from tqdm.auto import tqdm

# Modern OpenAI SDK imports
try:
    from openai import OpenAI, APIStatusError, APITimeoutError
    # Map the name used in your reasoning code to the actual SDK class name
    APIStatusException = APIStatusError
except ImportError:
    print("❌ Failed to import modern OpenAI classes. Please ensure cell 1.1 finished successfully.")

# Retry delays
RETRY_DELAYS_TRANSIENT = [2, 5]
RETRY_DELAYS_QUOTA     = [30, 60]

print("✅ Libraries imported")

✅ Libraries imported


## Layer 2 — OCR (Typhoon API)

Output structure:
```
Textlog/
├── constituency/<doc_name>/<stem>.text
└── party_list/<doc_name>/<stem>.text
```

In [5]:
# ── 2.1 Scan & Organize Image Files ──────────────────────────────────

def scan_images(images_dir: str) -> dict:
    """
    Scan images directory and group by document.
    Returns:
      {
        'constituency': { 'constituency_10_20': ['/path/to/page1.png', ...], ... },
        'party_list':   { 'party_list_30_2':    ['/path/to/page1.png', ...], ... }
      }
    """
    all_pngs = sorted(glob.glob(os.path.join(images_dir, '*.png')))
    print(f"Found {len(all_pngs)} PNG files")

    groups = {'constituency': defaultdict(list), 'party_list': defaultdict(list)}

    for fpath in all_pngs:
        fname = os.path.basename(fpath)
        if fname.startswith('constituency_'):
            category = 'constituency'
        elif fname.startswith('party_list_'):
            category = 'party_list'
        else:
            continue

        # constituency_10_20.png       → constituency_10_20
        # constituency_10_20_page2.png → constituency_10_20
        name_no_ext = os.path.splitext(fname)[0]
        base_name = re.sub(r'_page\d+$', '', name_no_ext)
        groups[category][base_name].append(fpath)

    for cat in groups:
        for key in groups[cat]:
            groups[cat][key] = sorted(groups[cat][key])

    print(f"  Constituency: {len(groups['constituency'])} documents")
    print(f"  Party list:   {len(groups['party_list'])} documents")
    return groups


image_groups = scan_images(DIRS['images'])

Found 847 PNG files
  Constituency: 150 documents
  Party list:   150 documents


In [6]:
# ── 2.2 Typhoon OCR API ───────────────────────────────────────────────

def extract_text_typhoon_api(image_path: str) -> str:
    """
    Send image to Typhoon OCR API and return extracted text.
    Raises on API error.
    """
    if not TYPHOON_API_KEY:
        raise RuntimeError("TYPHOON_API_KEY not configured")

    url = "https://api.opentyphoon.ai/v1/ocr"
    headers = {'Authorization': f'Bearer {TYPHOON_API_KEY}'}
    data = {
        'model': 'typhoon-ocr',
        'task_type': 'default',
        'max_tokens': 16384,
        'temperature': 0.1,
        'top_p': 0.6,
        'repetition_penalty': 1.2,
    }

    with open(image_path, 'rb') as f:
        response = requests.post(
            url,
            files={'file': f},
            data=data,
            headers=headers,
        )

    response.raise_for_status()
    result = response.json()

    extracted_texts = []
    for page_result in result.get('results', []):
        if page_result.get('success'):
            content = page_result['message']['choices'][0]['message']['content']
            extracted_texts.append(content)

    if not extracted_texts:
        raise ValueError(f"No successful OCR results for {image_path}")

    return '\n'.join(extracted_texts)


print("✅ Typhoon OCR API ready")

✅ Typhoon OCR API ready


In [ ]:
# ── 2.3 Run OCR Batch ─────────────────────────────────────────────────

OCR_CALL_DELAY = 0.5  # วินาที ระหว่าง API calls


def run_ocr_batch(image_groups: dict, skip_existing: bool = True) -> dict:
    if not TYPHOON_API_KEY:
        print("❌ TYPHOON_API_KEY not configured — ยกเลิก")
        return {'processed': 0, 'skipped': 0, 'errors': 1}

    stats = {'processed': 0, 'skipped': 0, 'errors': 0}

    for category in ['constituency', 'party_list']:
        docs = image_groups[category]
        print(f"\n{'='*60}")
        print(f"OCR: {category} — {len(docs)} documents")
        print(f"{'='*60}")

        for doc_name, pages in tqdm(docs.items(), desc=category):
            subfolder = os.path.join(DIRS['textlog'], category, doc_name)
            os.makedirs(subfolder, exist_ok=True)

            for img_path in pages:
                stem = os.path.splitext(os.path.basename(img_path))[0]
                out_path = os.path.join(subfolder, f"{stem}.text")

                if skip_existing and os.path.exists(out_path):
                    stats['skipped'] += 1
                    continue

                try:
                    text = extract_text_typhoon_api(img_path)
                    with open(out_path, 'w', encoding='utf-8') as f:
                        f.write(text)
                    stats['processed'] += 1
                    time.sleep(OCR_CALL_DELAY)
                except Exception as e:
                    print(f"  ❌ [{stem}]: {e}")
                    stats['errors'] += 1

    print(f"\n{'='*60}")
    print(f"✅ Done! Processed: {stats['processed']} | Skipped: {stats['skipped']} | Errors: {stats['errors']}")
    return stats


ocr_stats = run_ocr_batch(image_groups, skip_existing=True)

In [6]:
# ── 2.4 (Optional) ตรวจสอบผลลัพธ์ OCR ───────────────────────────────

const_logs = glob.glob(os.path.join(DIRS['textlog'], 'constituency', '**', '*.text'), recursive=True)
party_logs = glob.glob(os.path.join(DIRS['textlog'], 'party_list',   '**', '*.text'), recursive=True)

print(f"Textlog Summary")
print(f"  Constituency : {len(const_logs)} files")
print(f"  Party List   : {len(party_logs)} files")
print(f"  Total        : {len(const_logs) + len(party_logs)} files")

for tf in (const_logs + party_logs)[:3]:
    print(f"\n{'─'*50}")
    print(f"📄 {os.path.relpath(tf, DIRS['textlog'])}")
    print(f"{'─'*50}")
    with open(tf, 'r', encoding='utf-8') as f:
        content = f.read()
    print(content[:500])
    print(f"  ... ({len(content)} chars total)")

Textlog Summary
  Constituency : 315 files
  Party List   : 532 files
  Total        : 847 files

──────────────────────────────────────────────────
📄 constituency/constituency_10_1/constituency_10_1.png.text
──────────────────────────────────────────────────
ส.ส. ๖/๑

ประกาศคณะกรรมการการเลือกตั้งประจำเขตเลือกตั้งที่ ๑
กรุงเทพมหานคร
เรื่อง ผลการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบแบ่งเขตเลือกตั้ง

ตามที่ได้มีพระราชกฤษฎีกาให้มีการเลือกตั้งสมาชิกสภาผู้แทนราษฎร และคณะกรรมการการเลือกตั้งได้กำหนดให้วันที่ ๘ เดือน กุมภาพันธ์ พ.ศ. ๒๕๖๙ เป็นวันเลือกตั้ง นั้น

บัดนี้ คณะกรรมการการเลือกตั้งประจำเขตเลือกตั้งที่ ๑ กรุงเทพมหานคร ได้ดำเนินการรวมผลคะแนนสมาชิกสภาผู้แทนราษฎรแบบแบ่งเขตเลือกตั้งเสร็จสิ้นเป็นที่เรียบร้อยแล้ว ดังนั้น จึงขอประกาศผลการนับคะแนนสมาชิกสภาผู้แทนราษ
  ... (1614 chars total)

──────────────────────────────────────────────────
📄 constituency/constituency_10_1/constituency_10_1_page3.png.text
──────────────────────────────────────────────────
ส.ส. ๖/๑
- ๓ -

ประกาศ ณ วันที่ ๘ เดือน กุมภา

## Layer 3 — Reasoning (DeepSeek V3.2)

อ่าน `.text` files จาก Textlog → DeepSeek extract `Id, Vote` → บันทึก CSV

```
Result/
├── constituency_10_20.csv
├── constituency_10_21.csv
└── party_list_30_7.csv
```

แต่ละไฟล์:
```csv
Id,Vote
constituency_10_20_1,1245
constituency_10_20_2,892
```

In [7]:
# ── 3.1 DeepSeek API Setup ────────────────────────────────────────────

DEEPSEEK_CALL_DELAY = 1.0  # วินาที ระหว่าง docs

_deepseek_client = None # Cache the client instance

def call_deepseek(prompt: str, max_retries: int = 2) -> str:
    """
    Call DeepSeek V3.2 (deepseek-chat) using OpenAI client.
    Retries on transient errors and timeouts.
    """
    global _deepseek_client

    if not DEEPSEEK_API_KEY:
        raise RuntimeError("DEEPSEEK_API_KEY not configured")

    if _deepseek_client is None:
        # DeepSeek's base_url for OpenAI client usually ends with /v1
        _deepseek_client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com/v1")

    for attempt in range(max_retries + 1):
        try:
            response = _deepseek_client.chat.completions.create(
                model="deepseek-chat",  # DeepSeek-V3.2
                max_tokens=512,          # CSV output สั้น ประหยัด output tokens
                messages=[{"role": "user", "content": prompt}],
                timeout=60,
            )
            return response.choices[0].message.content.strip()

        except APIStatusException as e:
            status = e.status_code
            is_last = attempt == max_retries

            # Don't retry auth/bad-request errors (401, 400, 403)
            if status in (401, 400, 403):
                print(f"  ❌ DeepSeek API authentication/bad request error (status {status}): {e}")
                raise # Re-raise immediately for auth/bad request errors

            if is_last:
                print(f"  ❌ DeepSeek API failed after {max_retries + 1} attempts (status {status}): {e}")
                raise

            delay = RETRY_DELAYS_TRANSIENT[min(attempt, len(RETRY_DELAYS_TRANSIENT) - 1)]
            print(f"  ⚠ DeepSeek API HTTP {status} on attempt {attempt + 1}, retrying in {delay}s…")
            time.sleep(delay)

        except APITimeoutError:
            if attempt == max_retries:
                print(f"  ❌ DeepSeek API timeout after {max_retries + 1} attempts.")
                raise
            delay = RETRY_DELAYS_TRANSIENT[min(attempt, len(RETRY_DELAYS_TRANSIENT) - 1)]
            print(f"  ⚠ DeepSeek API timeout on attempt {attempt + 1}, retrying in {delay}s…")
            time.sleep(delay)

        except Exception as e: # Catch any other unexpected errors
            if attempt == max_retries:
                print(f"  ❌ DeepSeek API unexpected error after {max_retries + 1} attempts: {e}")
                raise
            delay = RETRY_DELAYS_TRANSIENT[min(attempt, len(RETRY_DELAYS_TRANSIENT) - 1)]
            print(f"  ⚠ DeepSeek API unexpected error on attempt {attempt + 1}: {e}, retrying in {delay}s…")
            time.sleep(delay)


print("✅ DeepSeek V3.2 API ready")

✅ DeepSeek V3.2 API ready


In [8]:
# ── 3.2 Build Text Groups from Textlog Folder ─────────────────────────

def build_text_groups_from_folder(textlog_dir: str) -> dict:
    """
    Scan Textlog folder โดยตรง (ไม่ขึ้นกับ image_groups ใน memory)
    Returns: {
      'constituency': { doc_name: [(stem, text_path), ...] },
      'party_list':   { doc_name: [(stem, text_path), ...] }
    }
    """
    groups = {"constituency": defaultdict(list), "party_list": defaultdict(list)}

    for category in ["constituency", "party_list"]:
        pattern = os.path.join(textlog_dir, category, "**", "*.text")
        for text_path in sorted(glob.glob(pattern, recursive=True)):
            stem     = os.path.splitext(os.path.basename(text_path))[0]
            doc_name = os.path.basename(os.path.dirname(text_path))
            groups[category][doc_name].append((stem, text_path))

    for cat in groups:
        for doc in groups[cat]:
            groups[cat][doc] = sorted(groups[cat][doc])

    print("Text groups from Textlog:")
    print(f"  Constituency : {len(groups['constituency'])} documents")
    print(f"  Party List   : {len(groups['party_list'])} documents")
    return groups


text_groups = build_text_groups_from_folder(DIRS["textlog"])

Text groups from Textlog:
  Constituency : 150 documents
  Party List   : 150 documents


In [9]:
# ── 3.3 Reasoning & CSV Parsing ───────────────────────────────────────

def build_reasoning_prompt(doc_name: str, text_pages: list) -> str:
    combined_text = "\n".join(
        f"--- {page_name} ---\n{page_text}"
        for page_name, page_text in text_pages
    )
    example_rows = "\n".join([f"{doc_name}_{i}" for i in range(1, 4)])
    return (
        f"Extract vote scores from the OCR text.\n"
        f"Document: {doc_name}\n"
        f"OCR Text:\n{combined_text}\n"
        f"Instructions:\n"
        f"1. Find table/list with candidate numbers and vote scores.\n"
        f"2. Id format (candidate number appended to doc name):\n{example_rows}\n"
        f"3. Convert Thai numerals/text to Arabic numbers.\n"
        f"Reply with CSV only (Id,Vote) — no header, no explanation:"
    )


def parse_csv_response(raw: str, doc_name: str) -> list[dict]:
    """
    Parse LLM CSV response → list of {Id, Vote} dicts.
    Skips header row, blank lines, and malformed rows.
    """
    rows = []
    for line in raw.strip().splitlines():
        line = line.strip().strip('`')
        if not line or line.lower().startswith('id'):
            continue
        parts = line.split(',')
        if len(parts) < 2:
            continue
        candidate_id = parts[0].strip()
        vote_raw     = parts[1].strip()
        try:
            vote = int(float(vote_raw))
        except ValueError:
            continue
        rows.append({'Id': candidate_id, 'Vote': vote})
    return rows


def _load_text_pages(
    page_entries: list[tuple[str, str]],
    doc_name: str,
) -> tuple[list[tuple[str, str]], list[str]]:
    """Read all pages for a document. Returns (loaded_pages, failed_stems)."""
    text_pages, failed = [], []
    for page_stem, text_path in page_entries:
        try:
            with open(text_path, 'r', encoding='utf-8') as f:
                text_pages.append((page_stem, f.read()))
        except Exception as e:
            print(f"  ⚠  Cannot read {page_stem}: {e}")
            failed.append(page_stem)
    return text_pages, failed


print("✅ Reasoning functions ready")

✅ Reasoning functions ready


In [10]:
# ── 3.4 Run Reasoning Batch ───────────────────────────────────────────

def run_reasoning_batch(text_groups: dict, skip_existing: bool = True) -> dict:
    if not DEEPSEEK_API_KEY:
        print("❌ DEEPSEEK_API_KEY not configured — ยกเลิก")
        return {'processed': 0, 'skipped': 0, 'errors': 1}

    os.makedirs(DIRS['result'], exist_ok=True)
    stats = {'processed': 0, 'skipped': 0, 'errors': 0}

    for category in ['constituency', 'party_list']:
        docs = text_groups[category]
        print(f"\n{'='*60}")
        print(f"Reasoning: {category} — {len(docs)} documents")
        print(f"{'='*60}")

        for doc_name, page_entries in tqdm(docs.items(), desc=f"{category} reasoning"):
            csv_path = os.path.join(DIRS['result'], f"{doc_name}.csv")

            if skip_existing and os.path.exists(csv_path):
                stats['skipped'] += 1
                continue

            # 1. Load text pages
            text_pages, failed_pages = _load_text_pages(page_entries, doc_name)

            if not text_pages:
                print(f"  ⚠  No text for {doc_name} — skipping")
                stats['errors'] += 1
                continue

            if failed_pages:
                print(f"  ⚠  {doc_name}: {len(failed_pages)} page(s) missing ({', '.join(failed_pages)})")

            # 2. Call DeepSeek
            try:
                prompt = build_reasoning_prompt(doc_name, text_pages)
                raw_response = call_deepseek(prompt)
            except Exception as e:
                print(f"  ❌ DeepSeek failed for {doc_name}: {e}")
                stats['errors'] += 1
                time.sleep(DEEPSEEK_CALL_DELAY)
                continue

            # 3. Parse and save
            rows = parse_csv_response(raw_response, doc_name)
            if not rows:
                print(f"  ⚠  No votes parsed for {doc_name} (raw: {raw_response[:80]!r})")
                stats['errors'] += 1
            else:
                pd.DataFrame(rows).to_csv(csv_path, index=False)
                stats['processed'] += 1

            time.sleep(DEEPSEEK_CALL_DELAY)

    print(f"\n{'='*60}")
    print(
        f"✅ Done! Processed: {stats['processed']} | "
        f"Skipped: {stats['skipped']} | Errors: {stats['errors']}"
    )
    return stats


reason_stats = run_reasoning_batch(text_groups, skip_existing=True)


Reasoning: constituency — 150 documents


constituency reasoning:   0%|          | 0/150 [00:00<?, ?it/s]


Reasoning: party_list — 150 documents


party_list reasoning:   0%|          | 0/150 [00:00<?, ?it/s]


✅ Done! Processed: 279 | Skipped: 21 | Errors: 0


In [11]:
# ── 3.5 (Optional) ตรวจสอบ CSV ผลลัพธ์ ──────────────────────────────

csv_files = sorted(glob.glob(os.path.join(DIRS['result'], '*.csv')))
print(f"Total CSV files in Result/: {len(csv_files)}\n")

for cf in csv_files[:5]:
    print(f"📊 {os.path.basename(cf)}")
    df = pd.read_csv(cf)
    print(df.to_string(index=False))
    print()

Total CSV files in Result/: 300

📊 constituency_10_1.csv
                  Id  Vote
 constituency_10_1_1 34167
 constituency_10_1_2 14813
 constituency_10_1_3 14368
 constituency_10_1_4  6030
 constituency_10_1_5  2075
 constituency_10_1_6  1133
 constituency_10_1_7  1023
 constituency_10_1_8   979
 constituency_10_1_9   629
constituency_10_1_10   489
constituency_10_1_11   351
constituency_10_1_12   244
constituency_10_1_13   168
constituency_10_1_14   165
constituency_10_1_15   154
constituency_10_1_16   113
constituency_10_1_17    94
constituency_10_1_18    80

📊 constituency_10_10.csv
                   Id  Vote
 constituency_10_10_1 41804
 constituency_10_10_2 19047
 constituency_10_10_3  9440
 constituency_10_10_4  7925
 constituency_10_10_5  6372
 constituency_10_10_6  2012
 constituency_10_10_7  1437
 constituency_10_10_8   636
 constituency_10_10_9   583
constituency_10_10_10   544
constituency_10_10_11   503
constituency_10_10_12   495
constituency_10_10_13   461
constituency

## Layer 4 — Map Results to Submission Template


In [12]:
# ── 4.1 Load Submission Template ─────────────────────────────────────

template_path = os.path.join(ROOT, 'submission_template.csv')
template_df = pd.read_csv(template_path)

print(f"Template: {template_df.shape[0]} rows × {template_df.shape[1]} cols")
print(f"Columns:  {list(template_df.columns)}")
template_df.head(10)

Template: 10053 rows × 5 cols
Columns:  ['id', 'doc_id', 'row_num', 'party_name', 'votes']


,id,doc_id,row_num,party_name,votes
0,constituency_10_1_1,constituency_10_1,1,ประชาธิปัตย์,0
1,constituency_10_1_2,constituency_10_1,2,ภูมิใจไทย,0
2,constituency_10_1_3,constituency_10_1,3,เศรษฐกิจ,0
3,constituency_10_1_4,constituency_10_1,4,กล้าธรรม,0
4,constituency_10_1_5,constituency_10_1,5,พลวัต,0
5,constituency_10_1_6,constituency_10_1,6,ประชาชน,0
6,constituency_10_1_7,constituency_10_1,7,เพื่อไทย,0
7,constituency_10_1_8,constituency_10_1,8,ไทยภักดี,0
8,constituency_10_1_9,constituency_10_1,9,รวมไทยสร้างชาติ,0
9,constituency_10_1_10,constituency_10_1,10,ปวงชนไทย,0


In [14]:
# ── 4.2 Merge All Results ────────────────────────────────────────────

def merge_all_results(result_dir: str, template_df: pd.DataFrame) -> pd.DataFrame:
    """
    รวม CSV ทั้งหมดจาก Result/ แล้ว merge กับ submission template
    """
    csv_files = sorted(glob.glob(os.path.join(result_dir, '*.csv')))
    print(f"Loading {len(csv_files)} result CSV files...")

    all_rows = []
    for cf in csv_files:
        try:
            df = pd.read_csv(cf)
            if 'Id' in df.columns and 'Vote' in df.columns:
                all_rows.append(df[['Id', 'Vote']])
        except Exception as e:
            print(f"  ⚠  Error reading {os.path.basename(cf)}: {e}")

    if not all_rows:
        print("❌ No result CSVs found — returning empty template")
        return template_df

    results_df = pd.concat(all_rows, ignore_index=True)
    print(f"Total extracted results: {len(results_df)} rows")

    # Deduplicate — keep last occurrence
    dupes = results_df.duplicated(subset='Id', keep=False).sum()
    if dupes > 0:
        print(f"  ⚠  {dupes} duplicate Ids — keeping last")
        results_df = results_df.drop_duplicates(subset='Id', keep='last')

    # Merge into template
    id_col   = template_df.columns[0]
    vote_col = 'votes' if 'votes' in template_df.columns else template_df.columns[-1]

    # Fix: Ensure the target column is numeric to avoid Arrow/String dtype conflicts
    submission = template_df.copy()
    submission[vote_col] = pd.to_numeric(submission[vote_col], errors='coerce')

    vote_lookup = dict(zip(results_df['Id'].astype(str), results_df['Vote']))

    matched = 0
    for idx, row in submission.iterrows():
        tid = str(row[id_col])
        if tid in vote_lookup:
            submission.at[idx, vote_col] = vote_lookup[tid]
            matched += 1

    print(f"Matched {matched} / {len(submission)} rows ({matched/len(submission)*100:.1f}%)")

    # Coverage report
    template_ids = set(submission[id_col].astype(str))
    extra   = set(results_df['Id'].astype(str)) - template_ids
    missing = template_ids - set(results_df['Id'].astype(str))
    if extra:
        print(f"  ⚠  {len(extra)} result Ids not in template: {sorted(extra)[:10]}")
    if missing:
        print(f"  ⚠  {len(missing)} template Ids without results")

    return submission


submission_df = merge_all_results(DIRS['result'], template_df)
print(f"\nSubmission shape: {submission_df.shape}")
submission_df.head(20)

Loading 300 result CSV files...
Total extracted results: 8074 rows
  ⚠  40 duplicate Ids — keeping last
Matched 8037 / 10053 rows (79.9%)
  ⚠  17 result Ids not in template: ['constituency_12_4_11', 'constituency_13_3_9', 'constituency_13_4_10', 'constituency_13_4_11', 'constituency_13_5_12', 'constituency_14_2_18', 'constituency_14_5_8', 'constituency_15_2_5', 'constituency_16_3_10', 'constituency_16_3_11']
  ⚠  2016 template Ids without results

Submission shape: (10053, 5)


,id,doc_id,row_num,party_name,votes
0,constituency_10_1_1,constituency_10_1,1,ประชาธิปัตย์,34167
1,constituency_10_1_2,constituency_10_1,2,ภูมิใจไทย,14813
2,constituency_10_1_3,constituency_10_1,3,เศรษฐกิจ,14368
3,constituency_10_1_4,constituency_10_1,4,กล้าธรรม,6030
4,constituency_10_1_5,constituency_10_1,5,พลวัต,2075
5,constituency_10_1_6,constituency_10_1,6,ประชาชน,1133
6,constituency_10_1_7,constituency_10_1,7,เพื่อไทย,1023
7,constituency_10_1_8,constituency_10_1,8,ไทยภักดี,979
8,constituency_10_1_9,constituency_10_1,9,รวมไทยสร้างชาติ,629
9,constituency_10_1_10,constituency_10_1,10,ปวงชนไทย,489


In [15]:
# ── 4.3 Save Final Submission ────────────────────────────────────────

output_path = os.path.join(ROOT, 'submission.csv')
submission_df.to_csv(output_path, index=False)

print(f"✅ Submission saved → {output_path}")
print(f"   Rows:            {len(submission_df)}")
print(f"   Non-null votes:  {submission_df.iloc[:, 1].notna().sum()}")
print(f"\n--- Sample ---")
print(submission_df.head(10).to_string(index=False))

✅ Submission saved → /content/drive/MyDrive/Typhoon_OCR_Data/submission.csv
   Rows:            10053
   Non-null votes:  10053

--- Sample ---
                  id            doc_id  row_num      party_name  votes
 constituency_10_1_1 constituency_10_1        1    ประชาธิปัตย์  34167
 constituency_10_1_2 constituency_10_1        2       ภูมิใจไทย  14813
 constituency_10_1_3 constituency_10_1        3        เศรษฐกิจ  14368
 constituency_10_1_4 constituency_10_1        4        กล้าธรรม   6030
 constituency_10_1_5 constituency_10_1        5           พลวัต   2075
 constituency_10_1_6 constituency_10_1        6         ประชาชน   1133
 constituency_10_1_7 constituency_10_1        7        เพื่อไทย   1023
 constituency_10_1_8 constituency_10_1        8        ไทยภักดี    979
 constituency_10_1_9 constituency_10_1        9 รวมไทยสร้างชาติ    629
constituency_10_1_10 constituency_10_1       10        ปวงชนไทย    489


In [16]:
# ── 4.4 Pipeline Summary ─────────────────────────────────────────────

n_textlogs = len(glob.glob(os.path.join(DIRS['textlog'], '**/*.text'), recursive=True))
n_csvs     = len(glob.glob(os.path.join(DIRS['result'], '*.csv')))

print("📋 Pipeline Summary")
print(f"{'='*50}")
print(f"  OCR text logs:   {n_textlogs}")
print(f"  Result CSVs:     {n_csvs}")
print(f"  Submission rows: {len(submission_df)}")
print(f"  Matched votes:   {submission_df.iloc[:, 1].notna().sum()}")
print(f"\n  Output: {output_path}")

📋 Pipeline Summary
  OCR text logs:   847
  Result CSVs:     300
  Submission rows: 10053
  Matched votes:   10053

  Output: /content/drive/MyDrive/Typhoon_OCR_Data/submission.csv
